# Bank Marketing ML Project (Beginner Friendly)

This notebook combines the core project into one end-to-end workflow.

What you will learn step by step:
1. Load and validate the real Bank Marketing dataset
2. Explore the data with simple visualizations
3. Prepare data for machine learning using pipelines
4. Train and compare 9 classification models
5. Understand model behavior through metrics and plots
6. Make predictions for a new client profile

## Step 0 - Import Libraries

This project uses:
- `pandas`, `numpy` for data work
- `matplotlib`, `seaborn` for plots
- `scikit-learn` for ML pipelines and models

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

BASE_DIR = Path.cwd()
DATASET_PATH = BASE_DIR / 'data' / 'bank-additional-full.csv'
VISUALS_DIR = BASE_DIR / 'visualizations'
VISUALS_DIR.mkdir(exist_ok=True)

print('Working directory:', BASE_DIR)
print('Dataset path:', DATASET_PATH)

## Step 1 - Load the Dataset

If the local file is missing, this cell will try downloading it from a public mirror.

In [ ]:
import urllib.request

if not DATASET_PATH.exists():
    DATASET_PATH.parent.mkdir(exist_ok=True)
    url = 'https://raw.githubusercontent.com/pauloemmilio/Bank-Marketing-UCI/master/bank-additional-full.csv'
    print('Local dataset not found. Downloading...')
    urllib.request.urlretrieve(url, DATASET_PATH)
    print('Downloaded to:', DATASET_PATH)

df = pd.read_csv(DATASET_PATH, sep=';')

if 'y' not in df.columns:
    raise ValueError("Target column 'y' not found in dataset.")

print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

## Step 2 - Quick Data Understanding

We check:
- Data types
- Missing values
- Class balance of the target (`y`)

In [ ]:
print('Data types:')
display(df.dtypes)

print('Missing values per column:')
display(df.isna().sum())

print('Target distribution (counts):')
display(df['y'].value_counts())

print('Target distribution (%):')
display((df['y'].value_counts(normalize=True) * 100).round(2))

## Step 3 - Exploratory Data Analysis (EDA)

We create three core visualizations:
1. Target class distribution
2. Numeric feature distributions
3. Categorical feature distributions

In [ ]:
# 1) Target distribution
target_counts = df['y'].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target_counts.plot(kind='bar', ax=axes[0], color=['#D95F02', '#1B9E77'])
axes[0].set_title('Target Class Distribution')
axes[0].set_xlabel('Subscription Outcome')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

target_counts.plot(kind='pie', autopct='%1.1f%%', ax=axes[1], colors=['#D95F02', '#1B9E77'], startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Target Class Share')

fig.tight_layout()
fig.savefig(VISUALS_DIR / '01_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2) Numeric distributions
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
selected_numeric = numeric_columns[:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
for axis, column in zip(axes, selected_numeric):
    sns.histplot(df[column], kde=True, ax=axis, color='#4C78A8')
    axis.set_title(f'Distribution of {column}')

for axis in axes[len(selected_numeric):]:
    axis.axis('off')

fig.tight_layout()
fig.savefig(VISUALS_DIR / '02_numeric_features_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3) Categorical distributions
categorical_columns = [c for c in df.select_dtypes(include=['object', 'string']).columns if c != 'y']
selected_categorical = categorical_columns[:8]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.ravel()

for axis, column in zip(axes, selected_categorical):
    order = df[column].value_counts().index
    sns.countplot(data=df, x=column, order=order, ax=axis, color='#72B7B2')
    axis.set_title(f'{column} distribution')
    axis.set_xlabel('')
    axis.tick_params(axis='x', rotation=45)

for axis in axes[len(selected_categorical):]:
    axis.axis('off')

fig.tight_layout()
fig.savefig(VISUALS_DIR / '03_categorical_features_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 4 - Prepare Features and Build the Preprocessing Pipeline

Why pipeline?
- Numeric columns need imputation + scaling
- Categorical columns need imputation + one-hot encoding
- Pipeline avoids data leakage and keeps workflow clean

In [ ]:
df_model = df.copy()
df_model['y_binary'] = df_model['y'].map({'no': 0, 'yes': 1})

X = df_model.drop(columns=['y', 'y_binary'])
y = df_model['y_binary']

categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_cols),
    ('categorical', categorical_pipeline, categorical_cols)
])

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Numeric columns:', len(numeric_cols))
print('Categorical columns:', len(categorical_cols))

## Step 5 - Define 9 Models (Including Ensemble Models)

Models used:
- Logistic Regression
- Decision Tree
- Random Forest
- Support Vector Machine
- K-Nearest Neighbors
- Gradient Boosting
- Neural Network (MLP)
- Voting Ensemble
- Stacking Ensemble

In [ ]:
def build_models(preprocessor):
    return {
        'Logistic Regression': Pipeline([
            ('preprocessor', preprocessor),
            ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
        ]),
        'Decision Tree': Pipeline([
            ('preprocessor', preprocessor),
            ('model', DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, class_weight='balanced', random_state=42))
        ]),
        'Random Forest': Pipeline([
            ('preprocessor', preprocessor),
            ('model', RandomForestClassifier(n_estimators=250, min_samples_leaf=5, class_weight='balanced_subsample', random_state=42, n_jobs=-1))
        ]),
        'Support Vector Machine': Pipeline([
            ('preprocessor', preprocessor),
            ('model', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))
        ]),
        'K-Nearest Neighbors': Pipeline([
            ('preprocessor', preprocessor),
            ('model', KNeighborsClassifier(n_neighbors=15))
        ]),
        'Gradient Boosting': Pipeline([
            ('preprocessor', preprocessor),
            ('model', GradientBoostingClassifier(random_state=42))
        ]),
        'Neural Network (MLP)': Pipeline([
            ('preprocessor', preprocessor),
            ('model', MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', alpha=0.001, learning_rate='adaptive', max_iter=300, early_stopping=True, validation_fraction=0.1, random_state=42))
        ]),
        'Voting Ensemble': Pipeline([
            ('preprocessor', preprocessor),
            ('model', VotingClassifier(
                estimators=[
                    ('lr', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
                    ('rf', RandomForestClassifier(n_estimators=250, min_samples_leaf=5, class_weight='balanced_subsample', random_state=42, n_jobs=-1)),
                    ('gb', GradientBoostingClassifier(random_state=42)),
                    ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', alpha=0.001, learning_rate='adaptive', max_iter=300, early_stopping=True, validation_fraction=0.1, random_state=42))
                ],
                voting='soft',
                n_jobs=-1
            ))
        ]),
        'Stacking Ensemble': Pipeline([
            ('preprocessor', preprocessor),
            ('model', StackingClassifier(
                estimators=[
                    ('lr', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
                    ('rf', RandomForestClassifier(n_estimators=250, min_samples_leaf=5, class_weight='balanced_subsample', random_state=42, n_jobs=-1)),
                    ('gb', GradientBoostingClassifier(random_state=42))
                ],
                final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
                cv=3,
                n_jobs=-1
            ))
        ])
    }

models = build_models(preprocessor)
list(models.keys())

## Step 6 - Train and Evaluate All Models

We calculate:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC

In [ ]:
def probability_scores(model, features):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(features)[:, 1]
    if hasattr(model, 'decision_function'):
        s = model.decision_function(features)
        return (s - s.min()) / (s.max() - s.min() + 1e-9)
    raise AttributeError('Model has no probability or decision scores.')

evaluation_rows = []
detailed_results = {}
classification_reports = []

for model_name, pipeline in models.items():
    print(f'Training {model_name}...')
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = probability_scores(pipeline, X_test)

    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }
    evaluation_rows.append(metrics)

    detailed_results[model_name] = {
        'pipeline': pipeline,
        'predictions': y_pred,
        'probabilities': y_prob,
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred, target_names=['No Subscription', 'Subscription'], zero_division=0)
    }

    classification_reports.append(f"{model_name}\n{'-' * len(model_name)}\n{detailed_results[model_name]['classification_report']}\n")

results_df = pd.DataFrame(evaluation_rows).set_index('Model').astype(float)
results_df = results_df.sort_values(by='F1-Score', ascending=False)

results_df.to_csv(VISUALS_DIR / 'model_results_summary.csv')
(VISUALS_DIR / 'classification_reports.txt').write_text('\n'.join(classification_reports), encoding='utf-8')

display(results_df.round(4))

## Step 7 - Compare Model Performance Visually

These plots make model behavior easier to explain to non-technical audiences.

In [ ]:
# 1) Bar chart + heatmap
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
heatmap_data = results_df[metrics_to_plot].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
heatmap_data.plot(kind='bar', ax=axes[0], width=0.85)
axes[0].set_title('Performance Comparison Across Models')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=35)

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlGnBu', ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Metric Heatmap')

fig.tight_layout()
fig.savefig(VISUALS_DIR / '04_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2) Confusion matrices
n_models = len(results_df)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
axes = axes.ravel()

for axis, model_name in zip(axes, results_df.index.tolist()):
    cm = detailed_results[model_name]['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axis)
    axis.set_title(f"{model_name}\nF1 = {results_df.loc[model_name, 'F1-Score']:.3f}")
    axis.set_xlabel('Predicted')
    axis.set_ylabel('Actual')

for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

fig.tight_layout()
fig.savefig(VISUALS_DIR / '05_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3) ROC curves
fig, ax = plt.subplots(figsize=(11, 8))
color_cycle = ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e', '#e6ab02', '#a6761d', '#2166ac', '#b2182b']

for color, model_name in zip(color_cycle, results_df.index.tolist()):
    y_prob = detailed_results[model_name]['probabilities']
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{model_name} (AUC={results_df.loc[model_name, 'ROC-AUC']:.3f})")

ax.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
ax.set_title('ROC Curve Comparison')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(VISUALS_DIR / '06_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 8 - Feature Importance (Tree-Based Model)

Feature importance helps explain *why* a model is making predictions.

In [ ]:
top_tree_model = None
for candidate in ['Random Forest', 'Gradient Boosting', 'Decision Tree']:
    if candidate in detailed_results:
        top_tree_model = candidate
        break

if top_tree_model is None:
    raise ValueError('No tree-based model found.')

pipeline = detailed_results[top_tree_model]['pipeline']
preproc = pipeline.named_steps['preprocessor']
estimator = pipeline.named_steps['model']
feature_names = preproc.get_feature_names_out()

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': estimator.feature_importances_
}).sort_values(by='Importance', ascending=False)

top_features = feature_importance.head(15)

fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=top_features, x='Importance', y='Feature', color='#4C78A8', ax=ax)
ax.set_title(f'Top 15 Feature Importances from {top_tree_model}')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')

fig.tight_layout()
fig.savefig(VISUALS_DIR / '07_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

feature_importance.to_csv(VISUALS_DIR / 'feature_importance.csv', index=False)
feature_importance.head(10)

## Step 9 - Make Predictions for a New Client

You can modify the values below to test how all models behave for different client profiles.

In [ ]:
new_client = {
    'age': 38,
    'job': 'admin.',
    'marital': 'married',
    'education': 'university.degree',
    'default': 'no',
    'housing': 'yes',
    'loan': 'no',
    'contact': 'cellular',
    'month': 'may',
    'day_of_week': 'thu',
    'duration': 220,
    'campaign': 2,
    'pdays': 999,
    'previous': 0,
    'poutcome': 'nonexistent',
    'emp.var.rate': 1.1,
    'cons.price.idx': 93.994,
    'cons.conf.idx': -36.4,
    'euribor3m': 4.857,
    'nr.employed': 5191.0
}

new_client_df = pd.DataFrame([new_client])

pred_rows = []
for model_name, pipe in models.items():
    prob_yes = probability_scores(pipe, new_client_df)[0]
    pred = int(prob_yes >= 0.5)
    pred_rows.append({
        'Model': model_name,
        'Probability of Subscription (yes)': prob_yes,
        'Predicted Class': 'yes' if pred == 1 else 'no'
    })

pred_df = pd.DataFrame(pred_rows).set_index('Model').sort_values(
    by='Probability of Subscription (yes)', ascending=False
)
display(pred_df.style.format({'Probability of Subscription (yes)': '{:.4f}'}))

## Step 10 - Final Summary

In this notebook, you completed a full ML project:
- Loaded and validated real business data
- Performed EDA and understood class imbalance
- Built robust preprocessing pipelines
- Trained and compared 9 models
- Evaluated with multiple metrics (not just accuracy)
- Visualized model behavior and feature importance
- Predicted outcomes for a new client profile

If you want, the next step can be hyperparameter tuning for top models (Voting, Random Forest, Stacking).